
# Evidencia de Aprendizaje 3: Optimización de procesos de desarrollo

**Estudiante:** Miguel Steven Morales  
**Asignatura:** Programación para Análisis de Datos  
**Institución:** Institución Universitaria Digital de Antioquia  
**Actividad:** F35 - Evidencia de aprendizaje 3

## 1. Introducción

Este notebook documenta la continuidad del proyecto académico de análisis de información comercial mediante Web Scraping.

En la EA1 se utilizaron BeautifulSoup, Selenium y Scrapy para obtener información comercial a partir de una copia HTML local de una publicación del Apple iPhone 15 de 128 GB en Mercado Libre Colombia.

En la EA2 se incorporaron pruebas automatizadas, control de versiones y un flujo CI/CD mediante GitHub Actions.

En la EA3 se desarrolló una aplicación interactiva con Streamlit, se ampliaron las pruebas automatizadas a un total de 20 y se publicó la solución en Streamlit Community Cloud.

El presente notebook utiliza los archivos CSV generados previamente para realizar un análisis reproducible, sin efectuar nuevas consultas a Mercado Libre.


## 2. Objetivos

### Objetivo general

Optimizar el proyecto de análisis de información comercial mediante la validación de datos, las pruebas automatizadas, la integración continua y el despliegue de una aplicación web en la nube.

### Objetivos específicos

- Cargar y validar los resultados obtenidos mediante BeautifulSoup, Selenium y Scrapy.
- Comparar los datos comerciales almacenados en los archivos CSV.
- Documentar las pruebas automatizadas implementadas con pytest.
- Explicar el flujo de integración continua configurado mediante GitHub Actions.
- Presentar la aplicación Streamlit publicada en la nube.


In [1]:

from pathlib import Path
import pandas as pd

# Archivos generados durante la Evidencia de Aprendizaje 1.
archivos = {
    "BeautifulSoup": "resultados_beautifulsoup.csv",
    "Selenium": "resultados_selenium.csv",
    "Scrapy": "resultados_scrapy.csv",
}

# Directorio principal del proyecto.
directorio = Path.cwd()

# Cargar los resultados de los tres métodos.
resultados = {}

for metodo, nombre_archivo in archivos.items():
    ruta = directorio / nombre_archivo

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró {nombre_archivo} en {directorio}"
        )

    resultados[metodo] = pd.read_csv(ruta)

    print(
        f"{metodo}: "
        f"{len(resultados[metodo])} registros cargados"
    )


BeautifulSoup: 1 registros cargados
Selenium: 1 registros cargados
Scrapy: 1 registros cargados



## 3. Validación de los datos

Antes de realizar el análisis comparativo, se verifica que los tres archivos CSV contengan los seis atributos comerciales definidos durante la EA1.

También se comprueba que los registros no estén vacíos, que sus campos obligatorios tengan información y que los precios sean valores numéricos positivos.

Estas validaciones permiten identificar problemas en los datos antes de utilizarlos en la aplicación web.


In [2]:

columnas_requeridas = {
    "Producto",
    "Precio_COP",
    "Vendedor",
    "Estado",
    "Disponibilidad",
    "URL",
}

for metodo, datos in resultados.items():
    # Comprobar la estructura del archivo.
    faltantes = columnas_requeridas - set(datos.columns)
    assert not faltantes, (
        f"{metodo}: faltan las columnas {faltantes}"
    )

    # Comprobar que existan registros.
    assert not datos.empty, (
        f"{metodo}: el archivo no contiene registros"
    )

    # Comprobar que los campos obligatorios tengan valores.
    assert datos[list(columnas_requeridas)].notna().all().all(), (
        f"{metodo}: existen valores nulos"
    )

    # Comprobar que los precios sean numéricos y positivos.
    precios = pd.to_numeric(datos["Precio_COP"], errors="coerce")
    assert precios.notna().all() and (precios > 0).all(), (
        f"{metodo}: se encontraron precios inválidos"
    )

    print(f"{metodo}: validación satisfactoria")

print("\nLos tres conjuntos de datos superaron las validaciones.")


BeautifulSoup: validación satisfactoria
Selenium: validación satisfactoria
Scrapy: validación satisfactoria

Los tres conjuntos de datos superaron las validaciones.



## 4. Comparación de los resultados comerciales

Una vez validados los archivos CSV, se comparan los resultados obtenidos mediante BeautifulSoup, Selenium y Scrapy.

El análisis permite identificar el precio registrado por cada herramienta y comprobar la consistencia de la información extraída.

Los datos proceden de una única publicación comercial almacenada previamente en un archivo HTML local. Por tanto, esta comparación no representa una evaluación de precios en tiempo real.


In [3]:

# Construir una tabla con los resultados de cada método.
comparacion = []

for metodo, datos in resultados.items():
    comparacion.append({
        "Metodo": metodo,
        "Producto": datos.iloc[0]["Producto"],
        "Precio_COP": int(datos.iloc[0]["Precio_COP"]),
        "Vendedor": datos.iloc[0]["Vendedor"],
        "Registros": len(datos),
    })

tabla_comparativa = pd.DataFrame(comparacion)

display(tabla_comparativa)

# Comprobar si los tres métodos registraron el mismo precio.
precios_iguales = tabla_comparativa["Precio_COP"].nunique() == 1

print(
    "\n¿Los tres métodos registraron el mismo precio?",
    "Sí" if precios_iguales else "No"
)


,Metodo,Producto,Precio_COP,Vendedor,Registros
0,BeautifulSoup,Apple iPhone 15 128 GB Negro,2410891,CELUMOVIL,1
1,Selenium,Apple iPhone 15 128 GB Negro,2410891,CELUMOVIL,1
2,Scrapy,Apple iPhone 15 128 GB Negro,2410891,CELUMOVIL,1



¿Los tres métodos registraron el mismo precio? Sí



## 5. Desarrollo de la aplicación web con Streamlit

Durante la Evidencia de Aprendizaje 3 se desarrolló una aplicación web interactiva utilizando Python, pandas y Streamlit.

El archivo principal `app.py` permite:

- Cargar los resultados comerciales almacenados en los tres archivos CSV.
- Seleccionar el método de extracción: BeautifulSoup, Selenium o Scrapy.
- Presentar los registros y atributos comerciales disponibles.
- Visualizar una comparación de los precios obtenidos.
- Informar que los datos proceden de una copia HTML local y no representan precios en tiempo real.

La aplicación utiliza los resultados generados en EA1 y los presenta mediante una interfaz web, sin necesidad de realizar nuevamente el scraping.

### Ejecución local

Desde la terminal de VS Code, en la carpeta principal del proyecto:

```powershell
.\.venv\Scripts\python.exe -m streamlit run app.py
```

### Aplicación publicada

[Consultar aplicación Streamlit](https://ea1webscraping-jkjjq2ayqrjxjhhcebtjsf.streamlit.app/)

El código fuente está disponible en el archivo [app.py](https://github.com/Miguel-Mora/EA1_Web_Scraping/blob/main/app.py).



## 6. Pruebas automatizadas con pytest

El proyecto cuenta con 20 pruebas automatizadas:

- **13 pruebas de EA2:** validan la existencia, estructura, integridad y consistencia de los archivos CSV.
- **7 pruebas de EA3:** verifican el funcionamiento de la aplicación Streamlit, la selección de métodos y la presentación de resultados.

Las pruebas de la aplicación utilizan `streamlit.testing.v1.AppTest`, lo que permite comprobar sus componentes sin abrir manualmente un navegador.

### Ejecución de las pruebas

Desde PowerShell, en la carpeta del proyecto:

```powershell
.\.venv\Scripts\python.exe -m pytest tests/ -q
```

**Resultado verificado durante el desarrollo:** 20 pruebas aprobadas.

Los archivos de prueba se encuentran en:

- [tests/test_resultados.py](https://github.com/Miguel-Mora/EA1_Web_Scraping/blob/main/tests/test_resultados.py)
- [tests/test_app.py](https://github.com/Miguel-Mora/EA1_Web_Scraping/blob/main/tests/test_app.py)

Estas pruebas verifican la estructura de los datos y los componentes principales de la aplicación, pero no realizan nuevas extracciones desde Internet.



## 6. Pruebas automatizadas con pytest

El proyecto cuenta con 20 pruebas automatizadas:

- **13 pruebas de EA2:** validan la existencia, estructura, integridad y consistencia de los archivos CSV.
- **7 pruebas de EA3:** verifican el funcionamiento de la aplicación Streamlit, la selección de métodos y la presentación de resultados.

Las pruebas de la aplicación utilizan `streamlit.testing.v1.AppTest`, lo que permite comprobar sus componentes sin abrir manualmente un navegador.

### Ejecución de las pruebas

Desde PowerShell, en la carpeta del proyecto:

```powershell
.\.venv\Scripts\python.exe -m pytest tests/ -q
```

**Resultado verificado durante el desarrollo:** 20 pruebas aprobadas.

Los archivos de prueba se encuentran en:

- [tests/test_resultados.py](https://github.com/Miguel-Mora/EA1_Web_Scraping/blob/main/tests/test_resultados.py)
- [tests/test_app.py](https://github.com/Miguel-Mora/EA1_Web_Scraping/blob/main/tests/test_app.py)

Estas pruebas verifican la estructura de los datos y los componentes principales de la aplicación, pero no realizan nuevas extracciones desde Internet.



## 7. Integración continua y despliegue

### 7.1. Automatización con GitHub Actions

El proyecto utiliza GitHub Actions para ejecutar un flujo de integración continua definido en `.github/workflows/ci.yml`.

El trabajo de validación realiza las siguientes actividades:

1. Descarga el código desde GitHub.
2. Configura Python 3.11.
3. Instala las dependencias desde `requirements.txt`.
4. Ejecuta las 20 pruebas automatizadas.
5. Genera el reporte HTML mediante `generar_reporte.py`.
6. Verifica la existencia del archivo `site/index.html`.

El trabajo de publicación depende del resultado satisfactorio de las pruebas y se ejecuta cuando se realiza un `push` sobre `main`.

Después de integrar el pull request #2, la ejecución #10 de GitHub Actions terminó correctamente, con los trabajos de validación y publicación aprobados.

[Consultar configuración de GitHub Actions](https://github.com/Miguel-Mora/EA1_Web_Scraping/blob/main/.github/workflows/ci.yml)

### 7.2. Despliegue en GitHub Pages

El reporte HTML se publica mediante GitHub Actions:

[Consultar reporte HTML](https://miguel-mora.github.io/EA1_Web_Scraping/)

### 7.3. Despliegue en Streamlit Community Cloud

La aplicación interactiva se publicó utilizando Streamlit Community Cloud, conectando el repositorio de GitHub con el archivo principal `app.py`.

La configuración inicial utiliza la rama `feature/ea3-app`.

[Consultar aplicación interactiva](https://ea1webscraping-jkjjq2ayqrjxjhhcebtjsf.streamlit.app/)

El despliegue de GitHub Pages es ejecutado directamente por GitHub Actions, mientras que la publicación de la aplicación Streamlit es administrada por Streamlit Community Cloud.

Ambos servicios fueron comprobados después de integrar los cambios de EA3 en la rama principal.



## 8. Resultados y conclusiones

La Evidencia de Aprendizaje 3 permitió ampliar el proyecto de análisis de información comercial mediante una aplicación web interactiva y mecanismos de validación automatizada.

Los tres archivos CSV se utilizaron para consultar y comparar los resultados obtenidos con BeautifulSoup, Selenium y Scrapy. La información analizada corresponde a una publicación del Apple iPhone 15 de 128 GB y no representa datos comerciales en tiempo real.

La aplicación Streamlit facilitó la presentación de los resultados mediante tablas, indicadores y opciones de selección.

La incorporación de siete pruebas adicionales permitió alcanzar un total de 20 pruebas automatizadas. Estas se ejecutaron satisfactoriamente tanto en el entorno local como mediante GitHub Actions.

Asimismo, el proyecto conserva dos servicios publicados: un reporte HTML en GitHub Pages y una aplicación interactiva en Streamlit Community Cloud.

Como mejoras futuras se propone analizar múltiples publicaciones, incorporar procesos controlados de actualización de datos y ampliar las pruebas automatizadas.

### Repositorio del proyecto

[EA1_Web_Scraping — GitHub](https://github.com/Miguel-Mora/EA1_Web_Scraping)

### Limitaciones

Los resultados se obtuvieron a partir de una copia HTML local y representan únicamente una publicación comercial. Por tanto, no permiten realizar generalizaciones estadísticas sobre el mercado ni evaluar el rendimiento de las herramientas ante grandes volúmenes de datos.
